# spark initiation

In [1]:
!pip install pyspark -q
!pip install pyspark -q
!pip install langchain -q
!pip install langchain_openai -q
!pip install python-dotenv -q
!apt-get install openjdk-17-jdk-headless -qq > /dev/null

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

spark = (
    SparkSession.builder
    .appName("BT4221_HDB_Resale_Prices_Project")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.host", "localhost")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

print("Spark version:", spark.version)

spark.sparkContext.setLogLevel("WARN")

Spark version: 4.0.2


# Section 1: Dataset Extraction

## 1a. Hawker Centre Extraction

In [3]:
import os
import json
import logging
from typing import TypedDict, Literal

from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType, StructField, StructType, DoubleType, IntegerType,
)

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv()

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def load_secret(name: str, *alt_names: str) -> str:
    """Colab Secrets, or environment / `.env` when running locally."""
    try:
        from google.colab import userdata

        val = userdata.get(name)
        if val:
            return val
    except ImportError:
        pass
    try:
        from dotenv import load_dotenv

        load_dotenv(Path.cwd() / ".env")
    except ImportError:
        pass
    for key in (name, *alt_names):
        v = os.environ.get(key)
        if v:
            return v
    raise ValueError(
        f"Missing secret {name!r}. "
        "On Colab: add it under Secrets. Locally: set the env var or add it to `.env`."
    )

os.environ["SPARK_SUBMIT_OPTS"] = (
    "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
    "--add-opens java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens java.base/java.lang=ALL-UNNAMED "
    "--add-opens java.base/java.util=ALL-UNNAMED"
)

spark = (
    SparkSession.builder
    .appName("hawker_centre_extractor")
    .config("spark.driver.extraJavaOptions",
            "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
            "--add-opens java.base/sun.nio.ch=ALL-UNNAMED "
            "--add-opens java.base/java.lang=ALL-UNNAMED "
            "--add-opens java.base/java.util=ALL-UNNAMED "
            "-Djava.security.manager=allow")
    .config("spark.executor.extraJavaOptions",
            "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
            "-Djava.security.manager=allow")
    .getOrCreate()
)

llm = ChatOpenAI(
    model="gpt-5",
    temperature=0,
    api_key=load_secret("BT4221_OPENAI_API_KEY"),
)

SCRIPT_DIR = os.path.join(os.getcwd(), "dataset/hawker_centre")
GEOJSON_PATH = os.path.join(SCRIPT_DIR, "HawkerCentresGEOJSON.geojson")

class PipelineState(TypedDict):
    geojson_path: str
    raw_features: list[dict]        # raw GeoJSON features
    extraction_plan: dict           # agent's plan on which fields to extract
    extracted_data: list[dict]      # flat dicts after field extraction
    validated_data: list[dict]      # after quality checks
    transformed_data: list[dict]    # final cleaned records
    quality_report: dict
    agent_decision: str
    agent_reasoning: str
    retry_count: int
    output_path: str


def plan_extraction(state: PipelineState) -> PipelineState:
    """LLM agent inspects the GeoJSON schema and decides which fields to extract."""
    logger.info("Node: plan_extraction")

    # Read the GeoJSON and grab a sample feature for schema inspection
    with open(state["geojson_path"], "r") as f:
        geojson = json.load(f)

    features = geojson.get("features", [])
    state["raw_features"] = features

    if not features:
        state["agent_decision"] = "abort"
        state["agent_reasoning"] = "GeoJSON contains no features."
        return state

    sample = features[0]
    sample_props = sample.get("properties", {})
    sample_geom = sample.get("geometry", {})

    response = llm.invoke([
        SystemMessage(content=(
            "You are a data engineering agent for a Singapore HDB resale price prediction project. "
            "You are given the schema of a GeoJSON file containing hawker centre locations. "
            "Decide which fields are useful for the ML pipeline. "
            "The target variable is HDB resale_price. Hawker centre proximity and size are "
            "known predictors of resale value.\n\n"
            "IMPORTANT:\n"
            "- Do NOT include fields whose sample value is null or empty.\n"
            "- Latitude and longitude are ALREADY extracted from geometry coordinates — "
            "do NOT add any lat/lng/lon/latitude/longitude/point/x/y coordinate fields.\n"
            "- Only use source_keys that exist in the sample properties keys provided.\n\n"
            "Respond ONLY with valid JSON:\n"
            "{\n"
            '  "decision": "proceed" | "abort",\n'
            '  "reasoning": "...",\n'
            '  "fields_to_extract": [\n'
            '    {"source_key": "NAME", "target_column": "hawker_name", "dtype": "string"},\n'
            "    ...\n"
            "  ],\n"
            '  "filter_status": "Existing"  // or null if no filter\n'
            "}"
        )),
        HumanMessage(content=(
            f"GeoJSON name: {geojson.get('name', 'unknown')}\n"
            f"Total features: {len(features)}\n"
            f"Sample geometry: {json.dumps(sample_geom)}\n"
            f"Sample properties keys: {list(sample_props.keys())}\n"
            f"Sample properties values: {json.dumps(sample_props, default=str)}"
        )),
    ])

    try:
        plan = json.loads(response.content)
    except json.JSONDecodeError:
        logger.warning("Agent returned non-JSON, using default extraction plan.")
        plan = _default_extraction_plan()

    # Validate plan has required keys, fall back to defaults if malformed
    if not isinstance(plan.get("fields_to_extract"), list) or len(plan["fields_to_extract"]) == 0:
        logger.warning("Agent plan missing fields_to_extract, using defaults.")
        plan = _default_extraction_plan()

    state["extraction_plan"] = plan
    state["agent_decision"] = plan.get("decision", "proceed")
    state["agent_reasoning"] = plan.get("reasoning", "")

    logger.info(
        "Agent plan: extract %d fields, filter_status=%s — %s",
        len(plan["fields_to_extract"]),
        plan.get("filter_status"),
        plan.get("reasoning", ""),
    )
    return state


def _default_extraction_plan() -> dict:
    """Fallback extraction plan if the LLM response is unusable."""
    return {
        "decision": "proceed",
        "reasoning": "Default plan: extract core hawker centre fields for price prediction.",
        "filter_status": "Existing",
        "fields_to_extract": [
            {"source_key": "NAME", "target_column": "hawker_name", "dtype": "string"},
            {"source_key": "ADDRESSPOSTALCODE", "target_column": "hawker_postal_code", "dtype": "string"},
            {"source_key": "ADDRESSSTREETNAME", "target_column": "hawker_street", "dtype": "string"},
            {"source_key": "ADDRESSBLOCKHOUSENUMBER", "target_column": "hawker_block", "dtype": "string"},
            {"source_key": "ADDRESS_MYENV", "target_column": "hawker_address", "dtype": "string"},
            {"source_key": "NUMBER_OF_COOKED_FOOD_STALLS", "target_column": "hawker_num_stalls", "dtype": "int"},
            {"source_key": "STATUS", "target_column": "hawker_status", "dtype": "string"},
            {"source_key": "EST_ORIGINAL_COMPLETION_DATE", "target_column": "hawker_est_completion", "dtype": "string"},
        ],
    }


def extract_data(state: PipelineState) -> PipelineState:
    """Extract flat records from raw GeoJSON features based on the agent's plan."""
    logger.info("Node: extract_data")

    plan = state["extraction_plan"]
    field_map = plan["fields_to_extract"]
    filter_status = plan.get("filter_status")

    records = []
    skipped = 0

    for feature in state["raw_features"]:
        props = feature.get("properties", {})
        geom = feature.get("geometry", {})
        coords = geom.get("coordinates", [None, None])

        # Optional status filter
        if filter_status and props.get("STATUS", "") != filter_status:
            skipped += 1
            continue

        row = {
            "hawker_lat": coords[1] if len(coords) > 1 else None,
            "hawker_lng": coords[0] if len(coords) > 0 else None,
        }

        for fm in field_map:
            src = fm["source_key"]
            tgt = fm["target_column"]
            val = props.get(src)

            # Type coercion
            if fm.get("dtype") == "int" and val is not None:
                try:
                    val = int(val)
                except (ValueError, TypeError):
                    val = None
            elif fm.get("dtype") == "double" and val is not None:
                try:
                    val = float(val)
                except (ValueError, TypeError):
                    val = None

            row[tgt] = val

        records.append(row)

    state["extracted_data"] = records
    state["quality_report"] = {
        "total_features": len(state["raw_features"]),
        "extracted": len(records),
        "filtered_out": skipped,
        "filter_status": filter_status,
        "null_lat_count": sum(1 for r in records if r.get("hawker_lat") is None),
        "null_name_count": sum(1 for r in records if r.get("hawker_name") is None),
    }

    logger.info(
        "Extracted %d records (%d filtered out by status='%s')",
        len(records), skipped, filter_status,
    )
    return state


def validate_extraction(state: PipelineState) -> PipelineState:
    """LLM agent reviews extraction quality and decides next step."""
    logger.info("Node: validate_extraction")

    qr = state["quality_report"]

    response = llm.invoke([
        SystemMessage(content=(
            "You are a data quality agent. Evaluate the hawker centre extraction report. "
            "Respond ONLY with valid JSON:\n"
            '{"decision": "proceed"|"retry"|"abort", "reasoning": "..."}\n'
            "Rules:\n"
            "- If extracted >= 80% of total features (after valid filtering): proceed\n"
            "- If null_lat_count > 10% of extracted: retry (max 2 retries)\n"
            "- If extracted == 0: abort"
        )),
        HumanMessage(content=(
            f"Quality report: {json.dumps(qr)}. "
            f"Retry count: {state['retry_count']}. Max retries: 2."
        )),
    ])

    try:
        agent_output = json.loads(response.content)
    except json.JSONDecodeError:
        # Fallback rule-based decision
        if qr["extracted"] == 0:
            agent_output = {"decision": "abort", "reasoning": "No records extracted."}
        elif qr["null_lat_count"] / max(qr["extracted"], 1) > 0.1:
            agent_output = {"decision": "retry", "reasoning": "Too many null coordinates."}
        else:
            agent_output = {"decision": "proceed", "reasoning": "Fallback: quality looks acceptable."}

    state["agent_decision"] = agent_output["decision"]
    state["agent_reasoning"] = agent_output["reasoning"]

    if agent_output["decision"] == "retry":
        state["retry_count"] += 1

    logger.info("Agent QA Decision: %s — %s", agent_output["decision"], agent_output["reasoning"])
    return state


def transform_data(state: PipelineState) -> PipelineState:
    """Use PySpark to clean and finalize the extracted hawker centre data."""
    logger.info("Node: transform_data")

    records = state["extracted_data"]
    if not records:
        state["transformed_data"] = []
        return state

    # Infer columns from extraction plan + lat/lng
    columns = ["hawker_lat", "hawker_lng"] + [
        fm["target_column"] for fm in state["extraction_plan"]["fields_to_extract"]
    ]

    # Build PySpark schema dynamically
    type_map = {"string": StringType(), "int": IntegerType(), "double": DoubleType()}
    fields = [
        StructField("hawker_lat", DoubleType(), True),
        StructField("hawker_lng", DoubleType(), True),
    ]
    for fm in state["extraction_plan"]["fields_to_extract"]:
        spark_type = type_map.get(fm.get("dtype", "string"), StringType())
        fields.append(StructField(fm["target_column"], spark_type, True))

    schema = StructType(fields)

    # Create DataFrame
    rows = [{col: r.get(col) for col in columns} for r in records]
    df = spark.createDataFrame(rows, schema=schema)

    # Drop duplicates
    df = df.dropDuplicates()

    # Drop rows with null coordinates
    df = df.filter(F.col("hawker_lat").isNotNull() & F.col("hawker_lng").isNotNull())

    # Drop rows with null name
    if "hawker_name" in df.columns:
        df = df.filter(F.col("hawker_name").isNotNull())

    # Round coordinates to 8 decimal places for consistency
    df = df.withColumn("hawker_lat", F.round(F.col("hawker_lat"), 8))
    df = df.withColumn("hawker_lng", F.round(F.col("hawker_lng"), 8))

    state["transformed_data"] = [row.asDict() for row in df.collect()]
    logger.info("Transformed %d hawker centre records", len(state["transformed_data"]))
    return state

def decide_output(state: PipelineState) -> PipelineState:
    """Write final hawker centre data to CSV."""
    logger.info("Node: decide_output")

    output_dir = SCRIPT_DIR
    os.makedirs(output_dir, exist_ok=True)
    path = os.path.join(output_dir, "hawker_centres.csv")

    # Build explicit schema to avoid inference failures on all-null columns
    type_map = {"string": StringType(), "int": IntegerType(), "double": DoubleType()}
    fields = [
        StructField("hawker_lat", DoubleType(), True),
        StructField("hawker_lng", DoubleType(), True),
    ]
    for fm in state["extraction_plan"]["fields_to_extract"]:
        spark_type = type_map.get(fm.get("dtype", "string"), StringType())
        fields.append(StructField(fm["target_column"], spark_type, True))
    schema = StructType(fields)

    df = spark.createDataFrame(state["transformed_data"], schema=schema)

    # Reorder columns: name first, then location, then metadata
    preferred_order = [
        "hawker_name", "hawker_lat", "hawker_lng", "hawker_postal_code",
        "hawker_street", "hawker_block", "hawker_address",
        "hawker_num_stalls", "hawker_status", "hawker_est_completion",
    ]
    existing_cols = [c for c in preferred_order if c in df.columns]
    remaining_cols = [c for c in df.columns if c not in existing_cols]
    df = df.select(existing_cols + remaining_cols)

    df.toPandas().to_csv(path, index=False)

    state["output_path"] = path
    state["agent_reasoning"] = f"CSV output written to {path}"
    logger.info("Output written to %s (%d rows)", path, len(state["transformed_data"]))
    return state

def route_after_plan(state: PipelineState) -> Literal["extract_data", "__end__"]:
    if state["agent_decision"] == "abort":
        logger.warning("Agent aborted at planning: %s", state["agent_reasoning"])
        return END
    return "extract_data"


def route_after_validation(
    state: PipelineState,
) -> Literal["transform_data", "extract_data", "__end__"]:
    if state["agent_decision"] == "proceed":
        return "transform_data"
    elif state["agent_decision"] == "retry" and state["retry_count"] <= 2:
        return "extract_data"
    else:
        logger.warning("Agent aborted after validation: %s", state["agent_reasoning"])
        return END


# Build LangGraph Pipeline

workflow = StateGraph(PipelineState)

workflow.add_node("plan_extraction", plan_extraction)
workflow.add_node("extract_data", extract_data)
workflow.add_node("validate_extraction", validate_extraction)
workflow.add_node("transform_data", transform_data)
workflow.add_node("decide_output", decide_output)

workflow.set_entry_point("plan_extraction")
workflow.add_conditional_edges("plan_extraction", route_after_plan)
workflow.add_edge("extract_data", "validate_extraction")
workflow.add_conditional_edges("validate_extraction", route_after_validation)
workflow.add_edge("transform_data", "decide_output")
workflow.add_edge("decide_output", END)

app = workflow.compile()


## 1b. Shopping Mall Extraction

In [4]:
import os
import json
from dotenv import load_dotenv
import requests
import logging
import time
from typing import TypedDict, Literal, Optional
from bs4 import BeautifulSoup
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType, DoubleType

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv()

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

os.environ["SPARK_SUBMIT_OPTS"] = (
    "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
    "--add-opens java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens java.base/java.lang=ALL-UNNAMED "
    "--add-opens java.base/java.util=ALL-UNNAMED"
)

spark = (
    SparkSession.builder
    .appName("shopping_mall_extractor")
    .config("spark.driver.extraJavaOptions",
            "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
            "--add-opens java.base/sun.nio.ch=ALL-UNNAMED "
            "--add-opens java.base/java.lang=ALL-UNNAMED "
            "--add-opens java.base/java.util=ALL-UNNAMED "
            "-Djava.security.manager=allow")
    .config("spark.executor.extraJavaOptions",
            "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
            "-Djava.security.manager=allow")
    .getOrCreate()
)

llm = ChatOpenAI(
    model="gpt-5",
    temperature=0,
    api_key=load_secret("BT4221_OPENAI_API_KEY"),
)


class PipelineState(TypedDict):
    mall_names: list[str]
    raw_data: list[dict]
    validated_data: list[dict]
    transformed_data: list[dict]
    quality_report: dict
    agent_decision: str
    retry_count: int
    agent_reasoning: str
    output_path: str


def scrape_mall_names() -> list[str]:
    """Scrape shopping mall names from the Wikipedia list page using BeautifulSoup."""
    url = "https://en.wikipedia.org/wiki/List_of_shopping_malls_in_Singapore"
    headers = {"User-Agent": "Mozilla/5.0 (compatible; BT4221-Bot/1.0)"}

    resp = requests.get(url, headers=headers, timeout=15)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "lxml")

    # Target region headings: Central, East, North, North-East, West
    target_regions = {"Central", "East", "North", "North-East", "West"}
    all_malls = []

    content_div = soup.find("div", class_="mw-parser-output")
    if not content_div:
        logger.error("Could not find mw-parser-output div on Wikipedia page")
        return []

    # Iterate through direct children of the content div sequentially
    current_region = None
    for child in content_div.children:
        if not hasattr(child, "name") or child.name is None:
            continue

        # Check for heading div (region header)
        if child.name == "div" and "mw-heading2" in (child.get("class") or []):
            h2 = child.find("h2")
            if h2:
                region = h2.get("id") or h2.get_text(strip=True)
                if region in target_regions:
                    current_region = region
                else:
                    current_region = None
            continue

        # If we're in a target region, look for div-col with mall lists
        if current_region and child.name == "div" and "div-col" in (child.get("class") or []):
            for li in child.find_all("li"):
                mall_name = li.get_text(strip=True)
                if mall_name:
                    all_malls.append(mall_name)

    logger.info("Scraped %d mall entries from Wikipedia across %d regions",
                len(all_malls), len(target_regions))
    return all_malls


def plan_extraction(state: PipelineState) -> PipelineState:
    """Scrape Wikipedia for all shopping mall names in Singapore, then validate with agent."""
    # Web scrape mall names from Wikipedia
    try:
        scraped_malls = scrape_mall_names()
    except Exception as e:
        logger.error("Failed to scrape Wikipedia: %s", e)
        state["agent_decision"] = "abort"
        state["agent_reasoning"] = f"Wikipedia scrape failed: {e}"
        return state

    # Deduplicate by normalised name (case-insensitive)
    seen = set()
    unique_malls = []
    for m in scraped_malls:
        key = m.strip().lower()
        if key not in seen:
            seen.add(key)
            unique_malls.append(m.strip())

    # Agent validates the scraped list
    response = llm.invoke([
        SystemMessage(content=(
            "You are a data engineering agent. Evaluate the shopping mall list scraped from "
            "Wikipedia (https://en.wikipedia.org/wiki/List_of_shopping_malls_in_Singapore) and "
            "respond ONLY with valid JSON: "
            '{"decision": "proceed"|"abort", "reasoning": "...", "mall_count": <int>}'
        )),
        HumanMessage(content=(
            f"I scraped {len(unique_malls)} unique shopping malls from Wikipedia. "
            f"Sample: {unique_malls[:5]}...{unique_malls[-5:]}. "
            f"Should I proceed with OneMap geocoding extraction?"
        )),
    ])

    try:
        agent_output = json.loads(response.content)
    except json.JSONDecodeError:
        agent_output = {"decision": "proceed", "reasoning": "Defaulting to proceed", "mall_count": len(unique_malls)}

    # Override abort if count is within expected range
    if agent_output["decision"] == "abort" and 20 <= len(unique_malls) <= 500:
        logger.warning("Agent suggested abort but mall count looks valid (%d). Overriding to proceed.", len(unique_malls))
        agent_output["decision"] = "proceed"
        agent_output["reasoning"] = f"Overridden: {agent_output['reasoning']}"

    logger.info("Scraped %d total unique malls from Wikipedia", len(unique_malls))
    logger.info("Agent Plan Decision: %s — %s", agent_output["decision"], agent_output["reasoning"])

    state["mall_names"] = unique_malls
    state["agent_decision"] = agent_output["decision"]
    state["agent_reasoning"] = agent_output["reasoning"]
    return state


def _onemap_search(search_val: str, max_retries: int = 3) -> Optional[dict]:
    """Call OneMap search API with retry logic for rate limiting."""
    for attempt in range(max_retries):
        try:
            url = (
                f"https://www.onemap.gov.sg/api/common/elastic/search"
                f"?searchVal={search_val}&returnGeom=Y&getAddrDetails=Y&pageNum=1"
            )
            resp = requests.get(url, timeout=10)

            if resp.status_code == 429 or resp.status_code >= 500:
                wait = 2 ** attempt
                logger.info("Rate limited on '%s', retrying in %ds...", search_val, wait)
                time.sleep(wait)
                continue

            if resp.status_code != 200:
                return None

            ctype = resp.headers.get("Content-Type", "")
            if "application/json" not in ctype.lower():
                # Empty/non-JSON response often means rate limiting
                wait = 2 ** attempt
                logger.info("Non-JSON response for '%s', retrying in %ds...", search_val, wait)
                time.sleep(wait)
                continue

            return resp.json()

        except Exception as e:
            logger.warning("Request error for '%s' (attempt %d): %s", search_val, attempt + 1, e)
            time.sleep(2 ** attempt)

    return None


def extract_data(state: PipelineState) -> PipelineState:
    """Search OneMap using shopping mall names for geocoding results."""
    mall_data = []
    failed_names = []

    # Alias map: Wikipedia name -> list of alternative OneMap search terms
    alias_map = {
        "Holland Village Shopping Mall": ["Holland Village"],
        "Shaw House and Centre": ["Shaw House", "Shaw Centre"],
        "Novena Square Shopping Mall": ["Novena Square"],
        "i12 Katong": ["112 Katong", "I12 KATONG"],
        "Yew Tee Point": ["YEW TEE POINT", "Yew Tee"],
        "SingPostCentre": ["SingPost Centre", "Singapore Post Centre"],
        "Lot 1": ["Lot One Shoppers Mall", "LOT 1"],
    }

    for idx, name in enumerate(state["mall_names"]):
        # Build search term list: start with aliases if available, then the name itself
        if name in alias_map:
            search_terms = alias_map[name] + [name]
        else:
            search_terms = [name]

        # Add generic fallback variations
        if not any(kw in name.lower() for kw in ["mall", "shopping", "centre", "center", "plaza"]):
            search_terms.append(f"{name} Mall")
            search_terms.append(f"{name} Shopping Centre")

        found = False
        for search_val in search_terms:
            data = _onemap_search(search_val)
            if not data:
                continue

            if "results" in data and isinstance(data["results"], list) and data["results"]:
                best_result = None

                for result in data["results"]:
                    building = result.get("BUILDING", "").upper()
                    search_val_upper = result.get("SEARCHVAL", "").upper()
                    name_upper = name.upper()

                    # Priority 1: mall name appears in BUILDING or SEARCHVAL
                    if name_upper in building or name_upper in search_val_upper:
                        best_result = result
                        break

                    # Priority 2: keyword match (MALL, SHOPPING, PLAZA, etc.)
                    if not best_result and any(
                        kw in building or kw in search_val_upper
                        for kw in ["MALL", "SHOPPING", "PLAZA", "CENTRE", "CENTER",
                                   "POINT", "CITY", "SQUARE", "PLACE", "JUNCTION",
                                   "HUB", "LINK", "VILLAGE", "GALLERY"]
                    ):
                        best_result = result

                # Priority 3: fallback to the first result if no keyword match
                if not best_result:
                    best_result = data["results"][0]

                best_result["MALL_NAME"] = name
                mall_data.append(best_result)
                found = True

            if found:
                break

        if not found:
            logger.warning("Failed to find mall: %s", name)
            failed_names.append(name)

        # Throttle: 0.5s between requests to avoid rate limiting
        time.sleep(0.5)

        if (idx + 1) % 20 == 0:
            logger.info("Progress: %d / %d malls searched", idx + 1, len(state["mall_names"]))

    state["raw_data"] = mall_data
    state["quality_report"] = {
        "total_malls": len(state["mall_names"]),
        "extracted": len(mall_data),
        "failed": len(failed_names),
        "failed_names": failed_names[:20],
        "success_rate": round(len(mall_data) / max(len(state["mall_names"]), 1) * 100, 2),
    }
    logger.info("Extracted %d / %d malls", len(mall_data), len(state["mall_names"]))
    return state


def validate_extraction(state: PipelineState) -> PipelineState:
    """Agent reviews extraction quality and decides whether to proceed, retry, or abort."""
    qr = state["quality_report"]

    response = llm.invoke([
        SystemMessage(content=(
            "You are a data quality agent. Evaluate the extraction results and respond "
            "ONLY with valid JSON: "
            '{"decision": "proceed"|"retry"|"abort", "reasoning": "..."}'
            "\nRules: success_rate >= 80% → proceed; 50-80% → retry (max 2); <50% → abort."
        )),
        HumanMessage(content=(
            f"Extraction report: {json.dumps(qr)}. "
            f"Current retry count: {state['retry_count']}. "
            f"Max retries: 2. What should we do?"
        )),
    ])

    try:
        agent_output = json.loads(response.content)
    except json.JSONDecodeError:
        agent_output = {
            "decision": "proceed" if qr["success_rate"] >= 80 else "abort",
            "reasoning": "Fallback rule-based decision",
        }

    state["agent_decision"] = agent_output["decision"]
    state["agent_reasoning"] = agent_output["reasoning"]

    if agent_output["decision"] == "retry":
        state["retry_count"] += 1

    logger.info("Agent QA Decision: %s — %s", agent_output["decision"], agent_output["reasoning"])
    return state


def transform_data(state: PipelineState) -> PipelineState:
    """Transform raw OneMap results into a clean schema using PySpark."""
    schema = StructType([
        StructField("SEARCHVAL", StringType(), True),
        StructField("BLK_NO", StringType(), True),
        StructField("ROAD_NAME", StringType(), True),
        StructField("BUILDING", StringType(), True),
        StructField("ADDRESS", StringType(), True),
        StructField("POSTAL", StringType(), True),
        StructField("X", StringType(), True),
        StructField("Y", StringType(), True),
        StructField("LATITUDE", StringType(), True),
        StructField("LONGITUDE", StringType(), True),
        StructField("MALL_NAME", StringType(), True),
    ])

    df = spark.createDataFrame(state["raw_data"], schema=schema)
    df = df.dropDuplicates()
    df = df.drop("SEARCHVAL", "BLK_NO", "X", "Y")
    df = (
        df.withColumnRenamed("ROAD_NAME", "roadName")
          .withColumnRenamed("BUILDING", "building")
          .withColumnRenamed("ADDRESS", "address")
          .withColumnRenamed("POSTAL", "postalCode")
          .withColumnRenamed("LATITUDE", "latitude")
          .withColumnRenamed("LONGITUDE", "longitude")
          .withColumnRenamed("MALL_NAME", "mallName")
    )
    df = df.withColumn("latitude", F.col("latitude").cast(DoubleType()))
    df = df.withColumn("longitude", F.col("longitude").cast(DoubleType()))

    state["transformed_data"] = [row.asDict() for row in df.collect()]
    logger.info("Transformed %d mall records", df.count())
    return state


def decide_output(state: PipelineState) -> PipelineState:
    """Write cleaned shopping mall data to same directory as this script."""
    output_dir = os.path.join(os.getcwd(), "dataset/shopping_mall")
    os.makedirs(output_dir, exist_ok=True)

    path = os.path.join(output_dir, "shopping_malls.csv")

    df = spark.createDataFrame(state["transformed_data"])
    df.toPandas().to_csv(path, index=False)

    state["output_path"] = path
    state["agent_reasoning"] = "CSV output to dataset/shopping_mall."
    logger.info("Output written to %s", path)
    return state


def route_after_plan(state: PipelineState) -> Literal["extract_data", "__end__"]:
    if state["agent_decision"] == "abort":
        logger.warning("Agent aborted pipeline: %s", state["agent_reasoning"])
        return END
    return "extract_data"


def route_after_validation(state: PipelineState) -> Literal["transform_data", "extract_data", "__end__"]:
    if state["agent_decision"] == "proceed":
        return "transform_data"
    elif state["agent_decision"] == "retry" and state["retry_count"] <= 2:
        return "extract_data"
    else:
        logger.warning("Agent aborted after validation: %s", state["agent_reasoning"])
        return END


# Build LangGraph workflow
workflow = StateGraph(PipelineState)

workflow.add_node("plan_extraction", plan_extraction)
workflow.add_node("extract_data", extract_data)
workflow.add_node("validate_extraction", validate_extraction)
workflow.add_node("transform_data", transform_data)
workflow.add_node("decide_output", decide_output)

workflow.set_entry_point("plan_extraction")
workflow.add_conditional_edges("plan_extraction", route_after_plan)
workflow.add_edge("extract_data", "validate_extraction")
workflow.add_conditional_edges("validate_extraction", route_after_validation)
workflow.add_edge("transform_data", "decide_output")
workflow.add_edge("decide_output", END)

app = workflow.compile()

## 1c. Supermarket Extraction

In [5]:
import os
import re
import json
import logging
from typing import TypedDict, Literal

from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType, StructField, StructType, DoubleType, IntegerType,
)

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv()

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

os.environ["SPARK_SUBMIT_OPTS"] = (
    "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
    "--add-opens java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens java.base/java.lang=ALL-UNNAMED "
    "--add-opens java.base/java.util=ALL-UNNAMED"
)

spark = (
    SparkSession.builder
    .appName("supermarket_extractor")
    .config("spark.driver.extraJavaOptions",
            "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
            "--add-opens java.base/sun.nio.ch=ALL-UNNAMED "
            "--add-opens java.base/java.lang=ALL-UNNAMED "
            "--add-opens java.base/java.util=ALL-UNNAMED "
            "-Djava.security.manager=allow")
    .config("spark.executor.extraJavaOptions",
            "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
            "-Djava.security.manager=allow")
    .getOrCreate()
)

llm = ChatOpenAI(
    model="gpt-5",
    temperature=0,
    api_key=load_secret("BT4221_OPENAI_API_KEY"),
)

SCRIPT_DIR = os.path.join(os.getcwd(), "dataset/supermarket")
GEOJSON_PATH = os.path.join(SCRIPT_DIR, "SupermarketsGEOJSON.geojson")


# ---------------------------------------------------------------------------
# Utility: parse HTML description table into a flat dict
# ---------------------------------------------------------------------------
def _parse_html_description(html: str) -> dict:
    """Extract key-value pairs from the KML-style HTML table in Description."""
    pairs: dict[str, str] = {}
    # Each row looks like: <th>KEY</th> <td>VALUE</td>
    for match in re.finditer(
        r"<th>\s*([^<]+?)\s*</th>\s*<td>\s*([^<]*?)\s*</td>",
        html,
        re.IGNORECASE,
    ):
        key = match.group(1).strip()
        val = match.group(2).strip()
        if key.lower() != "attributes":  # skip the header row
            pairs[key] = val
    return pairs


# ---------------------------------------------------------------------------
# State definition
# ---------------------------------------------------------------------------
class PipelineState(TypedDict):
    geojson_path: str
    raw_features: list[dict]        # raw GeoJSON features
    extraction_plan: dict           # agent's plan on which fields to extract
    extracted_data: list[dict]      # flat dicts after field extraction
    validated_data: list[dict]      # after quality checks
    transformed_data: list[dict]    # final cleaned records
    quality_report: dict
    agent_decision: str
    agent_reasoning: str
    retry_count: int
    output_path: str


# ---------------------------------------------------------------------------
# Pipeline nodes
# ---------------------------------------------------------------------------
def plan_extraction(state: PipelineState) -> PipelineState:
    """LLM agent inspects the GeoJSON schema and decides which fields to extract."""
    logger.info("Node: plan_extraction")

    with open(state["geojson_path"], "r") as f:
        geojson = json.load(f)

    features = geojson.get("features", [])
    state["raw_features"] = features

    if not features:
        state["agent_decision"] = "abort"
        state["agent_reasoning"] = "GeoJSON contains no features."
        return state

    # Parse a sample feature so the LLM can inspect available keys
    sample = features[0]
    sample_geom = sample.get("geometry", {})
    sample_parsed = _parse_html_description(
        sample.get("properties", {}).get("Description", "")
    )

    response = llm.invoke([
        SystemMessage(content=(
            "You are a data engineering agent for a Singapore HDB resale price prediction project. "
            "You are given the schema of a GeoJSON file containing supermarket locations. "
            "The raw properties are embedded in an HTML table inside the 'Description' field "
            "and have already been parsed into key-value pairs for you.\n\n"
            "Decide which fields are useful for the ML pipeline. "
            "The target variable is HDB resale_price. Supermarket proximity is a "
            "known predictor of resale value.\n\n"
            "IMPORTANT:\n"
            "- Do NOT include fields whose sample value is null or empty.\n"
            "- Latitude and longitude are ALREADY extracted from geometry coordinates — "
            "do NOT add any lat/lng/lon/latitude/longitude fields.\n"
            "- Only use source_keys that appear in the parsed description keys provided.\n\n"
            "Respond ONLY with valid JSON:\n"
            "{\n"
            '  "decision": "proceed" | "abort",\n'
            '  "reasoning": "...",\n'
            '  "fields_to_extract": [\n'
            '    {"source_key": "LIC_NAME", "target_column": "supermarket_name", "dtype": "string"},\n'
            "    ...\n"
            "  ]\n"
            "}"
        )),
        HumanMessage(content=(
            f"Total features: {len(features)}\n"
            f"Sample geometry: {json.dumps(sample_geom)}\n"
            f"Parsed description keys: {list(sample_parsed.keys())}\n"
            f"Parsed description values: {json.dumps(sample_parsed, default=str)}"
        )),
    ])

    try:
        plan = json.loads(response.content)
    except json.JSONDecodeError:
        logger.warning("Agent returned non-JSON, using default extraction plan.")
        plan = _default_extraction_plan()

    if not isinstance(plan.get("fields_to_extract"), list) or len(plan["fields_to_extract"]) == 0:
        logger.warning("Agent plan missing fields_to_extract, using defaults.")
        plan = _default_extraction_plan()

    state["extraction_plan"] = plan
    state["agent_decision"] = plan.get("decision", "proceed")
    state["agent_reasoning"] = plan.get("reasoning", "")

    logger.info(
        "Agent plan: extract %d fields — %s",
        len(plan["fields_to_extract"]),
        plan.get("reasoning", ""),
    )
    return state


def _default_extraction_plan() -> dict:
    """Fallback extraction plan if the LLM response is unusable."""
    return {
        "decision": "proceed",
        "reasoning": "Default plan: extract core supermarket fields for price prediction.",
        "fields_to_extract": [
            {"source_key": "LIC_NAME", "target_column": "supermarket_name", "dtype": "string"},
            {"source_key": "BLK_HOUSE", "target_column": "supermarket_block", "dtype": "string"},
            {"source_key": "STR_NAME", "target_column": "supermarket_street", "dtype": "string"},
            {"source_key": "UNIT_NO", "target_column": "supermarket_unit", "dtype": "string"},
            {"source_key": "POSTCODE", "target_column": "supermarket_postal_code", "dtype": "string"},
            {"source_key": "LIC_NO", "target_column": "supermarket_licence_no", "dtype": "string"},
        ],
    }


def extract_data(state: PipelineState) -> PipelineState:
    """Extract flat records from raw GeoJSON features based on the agent's plan."""
    logger.info("Node: extract_data")

    plan = state["extraction_plan"]
    field_map = plan["fields_to_extract"]

    records: list[dict] = []

    for feature in state["raw_features"]:
        props = feature.get("properties", {})
        geom = feature.get("geometry", {})
        coords = geom.get("coordinates", [None, None])

        # Parse the HTML description into a flat dict
        parsed = _parse_html_description(props.get("Description", ""))

        row: dict = {
            "supermarket_lat": coords[1] if len(coords) > 1 else None,
            "supermarket_lng": coords[0] if len(coords) > 0 else None,
        }

        for fm in field_map:
            # Skip if the agent plan duplicates the hardcoded lat/lng columns
            if fm["target_column"] in ("supermarket_lat", "supermarket_lng"):
                continue
            src = fm["source_key"]
            tgt = fm["target_column"]
            val = parsed.get(src)

            # Type coercion
            if fm.get("dtype") == "int" and val is not None:
                try:
                    val = int(val)
                except (ValueError, TypeError):
                    val = None
            elif fm.get("dtype") == "double" and val is not None:
                try:
                    val = float(val)
                except (ValueError, TypeError):
                    val = None

            row[tgt] = val

        records.append(row)

    state["extracted_data"] = records
    state["quality_report"] = {
        "total_features": len(state["raw_features"]),
        "extracted": len(records),
        "null_lat_count": sum(1 for r in records if r.get("supermarket_lat") is None),
        "null_name_count": sum(1 for r in records if r.get("supermarket_name") is None),
    }

    logger.info("Extracted %d supermarket records", len(records))
    return state


def validate_extraction(state: PipelineState) -> PipelineState:
    """LLM agent reviews extraction quality and decides next step."""
    logger.info("Node: validate_extraction")

    qr = state["quality_report"]

    response = llm.invoke([
        SystemMessage(content=(
            "You are a data quality agent. Evaluate the supermarket extraction report. "
            "Respond ONLY with valid JSON:\n"
            '{"decision": "proceed"|"retry"|"abort", "reasoning": "..."}\n'
            "Rules:\n"
            "- If extracted >= 80% of total features: proceed\n"
            "- If null_lat_count > 10% of extracted: retry (max 2 retries)\n"
            "- If extracted == 0: abort"
        )),
        HumanMessage(content=(
            f"Quality report: {json.dumps(qr)}. "
            f"Retry count: {state['retry_count']}. Max retries: 2."
        )),
    ])

    try:
        agent_output = json.loads(response.content)
    except json.JSONDecodeError:
        if qr["extracted"] == 0:
            agent_output = {"decision": "abort", "reasoning": "No records extracted."}
        elif qr["null_lat_count"] / max(qr["extracted"], 1) > 0.1:
            agent_output = {"decision": "retry", "reasoning": "Too many null coordinates."}
        else:
            agent_output = {"decision": "proceed", "reasoning": "Fallback: quality looks acceptable."}

    state["agent_decision"] = agent_output["decision"]
    state["agent_reasoning"] = agent_output["reasoning"]

    if agent_output["decision"] == "retry":
        state["retry_count"] += 1

    logger.info("Agent QA Decision: %s — %s", agent_output["decision"], agent_output["reasoning"])
    return state


def transform_data(state: PipelineState) -> PipelineState:
    """Use PySpark to clean and finalize the extracted supermarket data."""
    logger.info("Node: transform_data")

    records = state["extracted_data"]
    if not records:
        state["transformed_data"] = []
        return state

    # Infer columns from extraction plan + lat/lng (exclude duplicates)
    plan_fields = [
        fm for fm in state["extraction_plan"]["fields_to_extract"]
        if fm["target_column"] not in ("supermarket_lat", "supermarket_lng")
    ]
    columns = ["supermarket_lat", "supermarket_lng"] + [
        fm["target_column"] for fm in plan_fields
    ]

    # Build PySpark schema dynamically
    type_map = {"string": StringType(), "int": IntegerType(), "double": DoubleType()}
    fields = [
        StructField("supermarket_lat", DoubleType(), True),
        StructField("supermarket_lng", DoubleType(), True),
    ]
    for fm in plan_fields:
        spark_type = type_map.get(fm.get("dtype", "string"), StringType())
        fields.append(StructField(fm["target_column"], spark_type, True))

    schema = StructType(fields)

    rows = [{col: r.get(col) for col in columns} for r in records]
    df = spark.createDataFrame(rows, schema=schema)

    # Drop duplicates
    df = df.dropDuplicates()

    # Drop rows with null coordinates
    df = df.filter(F.col("supermarket_lat").isNotNull() & F.col("supermarket_lng").isNotNull())

    # Drop rows with null name
    if "supermarket_name" in df.columns:
        df = df.filter(F.col("supermarket_name").isNotNull())

    # Round coordinates to 8 decimal places for consistency
    df = df.withColumn("supermarket_lat", F.round(F.col("supermarket_lat"), 8))
    df = df.withColumn("supermarket_lng", F.round(F.col("supermarket_lng"), 8))

    state["transformed_data"] = [row.asDict() for row in df.collect()]
    logger.info("Transformed %d supermarket records", len(state["transformed_data"]))
    return state


def decide_output(state: PipelineState) -> PipelineState:
    """Write final supermarket data to CSV."""
    logger.info("Node: decide_output")

    output_dir = SCRIPT_DIR
    os.makedirs(output_dir, exist_ok=True)
    path = os.path.join(output_dir, "supermarkets.csv")

    # Build explicit schema (exclude duplicates of lat/lng)
    plan_fields = [
        fm for fm in state["extraction_plan"]["fields_to_extract"]
        if fm["target_column"] not in ("supermarket_lat", "supermarket_lng")
    ]
    type_map = {"string": StringType(), "int": IntegerType(), "double": DoubleType()}
    fields = [
        StructField("supermarket_lat", DoubleType(), True),
        StructField("supermarket_lng", DoubleType(), True),
    ]
    for fm in plan_fields:
        spark_type = type_map.get(fm.get("dtype", "string"), StringType())
        fields.append(StructField(fm["target_column"], spark_type, True))
    schema = StructType(fields)

    df = spark.createDataFrame(state["transformed_data"], schema=schema)

    # Reorder columns: name first, then location, then metadata
    preferred_order = [
        "supermarket_name", "supermarket_lat", "supermarket_lng",
        "supermarket_postal_code", "supermarket_street", "supermarket_block",
        "supermarket_unit", "supermarket_licence_no",
    ]
    existing_cols = [c for c in preferred_order if c in df.columns]
    remaining_cols = [c for c in df.columns if c not in existing_cols]
    df = df.select(existing_cols + remaining_cols)

    df.toPandas().to_csv(path, index=False)

    state["output_path"] = path
    state["agent_reasoning"] = f"CSV output written to {path}"
    logger.info("Output written to %s (%d rows)", path, len(state["transformed_data"]))
    return state


# ---------------------------------------------------------------------------
# Routing functions
# ---------------------------------------------------------------------------
def route_after_plan(state: PipelineState) -> Literal["extract_data", "__end__"]:
    if state["agent_decision"] == "abort":
        logger.warning("Agent aborted at planning: %s", state["agent_reasoning"])
        return END
    return "extract_data"


def route_after_validation(
    state: PipelineState,
) -> Literal["transform_data", "extract_data", "__end__"]:
    if state["agent_decision"] == "proceed":
        return "transform_data"
    elif state["agent_decision"] == "retry" and state["retry_count"] <= 2:
        return "extract_data"
    else:
        logger.warning("Agent aborted after validation: %s", state["agent_reasoning"])
        return END


# ---------------------------------------------------------------------------
# Build LangGraph Pipeline
# ---------------------------------------------------------------------------
workflow = StateGraph(PipelineState)

workflow.add_node("plan_extraction", plan_extraction)
workflow.add_node("extract_data", extract_data)
workflow.add_node("validate_extraction", validate_extraction)
workflow.add_node("transform_data", transform_data)
workflow.add_node("decide_output", decide_output)

workflow.set_entry_point("plan_extraction")
workflow.add_conditional_edges("plan_extraction", route_after_plan)
workflow.add_edge("extract_data", "validate_extraction")
workflow.add_conditional_edges("validate_extraction", route_after_validation)
workflow.add_edge("transform_data", "decide_output")
workflow.add_edge("decide_output", END)

app = workflow.compile()

# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

## 1d. MRT/LRT Station Extraction

In [6]:
import os
import json
from dotenv import load_dotenv
import requests
import logging
import time
from typing import TypedDict, Literal
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType, DoubleType

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv()

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

os.environ["SPARK_SUBMIT_OPTS"] = (
    "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
    "--add-opens java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens java.base/java.lang=ALL-UNNAMED "
    "--add-opens java.base/java.util=ALL-UNNAMED"
)

spark = (
    SparkSession.builder
    .appName("mrt_lrt_stations_extractor")
    .config("spark.driver.extraJavaOptions",
            "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
            "--add-opens java.base/sun.nio.ch=ALL-UNNAMED "
            "--add-opens java.base/java.lang=ALL-UNNAMED "
            "--add-opens java.base/java.util=ALL-UNNAMED "
            "-Djava.security.manager=allow")
    .config("spark.executor.extraJavaOptions",
            "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
            "-Djava.security.manager=allow")
    .getOrCreate()
)

llm = ChatOpenAI(
    model="gpt-5",
    temperature=0,
    api_key=load_secret("BT4221_OPENAI_API_KEY"),
)

class PipelineState(TypedDict):
    station_codes: list[str]
    station_map: list[dict]  # [{"code": "NS1", "name": "Jurong East"}, ...]
    raw_data: list[dict]
    validated_data: list[dict]
    transformed_data: list[dict]
    quality_report: dict
    agent_decision: str
    retry_count: int
    agent_reasoning: str
    output_path: str


def plan_extraction(state: PipelineState) -> PipelineState:
    """Ask the LLM agent to generate all Singapore MRT/LRT station codes and names."""
    all_stations = []

    line_groups = [
        ("North South Line (NS)", "NS"),
        ("North East Line (NE)", "NE"),
        ("East West Line (EW) including Changi Extension (CG)", "EW, CG"),
        ("Circle Line (CC) including Circle Extension (CE)", "CC, CE"),
        ("Thomson-East Coast Line (TE)", "TE"),
        ("Downtown Line (DT)", "DT"),
        ("Bukit Panjang LRT (BP)", "BP"),
        ("Sengkang LRT (STC, SE, SW)", "STC, SE, SW"),
        ("Punggol LRT (PTC, PE, PW)", "PTC, PE, PW"),
    ]

    for line_name, prefixes in line_groups:
        response = llm.invoke([
            SystemMessage(content=(
                "You are a Singapore public transport expert. "
                "List ALL stations for the requested MRT/LRT line. "
                "Respond ONLY with a valid JSON array of objects. "
                'Each object must have "code" and "name" keys. '
                'Example: [{"code": "NS1", "name": "Jurong East"}, {"code": "NS2", "name": "Bukit Batok"}]. '
                "Include all currently operational stations as of 2026. "
                "Do NOT skip any stations."
            )),
            HumanMessage(content=(
                f"List every station code and name for the {line_name}. "
                f"Prefixes used: {prefixes}."
            )),
        ])

        try:
            stations = json.loads(response.content)
            if (isinstance(stations, list) and
                all(isinstance(s, dict) and "code" in s and "name" in s for s in stations)):
                all_stations.extend(stations)
                logger.info("Agent returned %d stations for %s", len(stations), line_name)
            else:
                logger.warning("Agent returned invalid format for %s: %s", line_name, response.content)
        except json.JSONDecodeError:
            logger.warning("Agent returned non-JSON for %s: %s", line_name, response.content)

    # Deduplicate by code
    seen = set()
    unique_stations = []
    for s in all_stations:
        if s["code"] not in seen:
            seen.add(s["code"])
            unique_stations.append(s)

    # Agent validates the combined list
    response = llm.invoke([
        SystemMessage(content=(
            "You are a data engineering agent. Evaluate the station list and "
            "respond ONLY with valid JSON: "
            '{"decision": "proceed"|"abort", "reasoning": "...", "code_count": <int>}'
        )),
        HumanMessage(content=(
            f"I have {len(unique_stations)} stations covering NSL, NEL, EWL, CCL, TEL, DTL, "
            f"BP LRT, Sengkang LRT, Punggol LRT. "
            f"Sample: {unique_stations[:3]}...{unique_stations[-3:]}. "
            f"Should I proceed with extraction?"
        )),
    ])

    try:
        agent_output = json.loads(response.content)
    except json.JSONDecodeError:
        agent_output = {"decision": "proceed", "reasoning": "Defaulting to proceed", "code_count": len(unique_stations)}

    # Override abort if codes are within expected range
    if agent_output["decision"] == "abort" and 150 <= len(unique_stations) <= 300:
        logger.warning("Agent suggested abort but stations look valid (%d). Overriding to proceed.", len(unique_stations))
        agent_output["decision"] = "proceed"
        agent_output["reasoning"] = f"Overridden: {agent_output['reasoning']}"

    logger.info("Agent generated %d total stations", len(unique_stations))
    logger.info("Agent Plan Decision: %s — %s", agent_output["decision"], agent_output["reasoning"])

    state["station_codes"] = [s["code"] for s in unique_stations]
    state["station_map"] = unique_stations
    state["agent_decision"] = agent_output["decision"]
    state["agent_reasoning"] = agent_output["reasoning"]
    return state


def extract_data(state: PipelineState) -> PipelineState:
    """Search OneMap using station name + MRT/LRT suffix for better hit rates."""
    mrt_lrt_data = []
    failed_codes = []

    # Build a code -> name lookup
    code_to_name = {s["code"]: s["name"] for s in state["station_map"]}

    for code in state["station_codes"]:
        station_name = code_to_name.get(code, code)

        # Try multiple search terms: "name MRT STATION", "name LRT STATION", then just code
        search_terms = [
            f"{station_name} MRT STATION",
            f"{station_name} LRT STATION",
            code,
        ]

        found = False
        for search_val in search_terms:
            try:
                url = (
                    f"https://www.onemap.gov.sg/api/common/elastic/search"
                    f"?searchVal={search_val}&returnGeom=Y&getAddrDetails=Y&pageNum=1"
                )
                resp = requests.get(url, timeout=10)

                if resp.status_code != 200:
                    continue

                ctype = resp.headers.get("Content-Type", "")
                if "application/json" not in ctype.lower():
                    continue

                data = resp.json()
                if "results" in data and isinstance(data["results"], list):
                    for result in data["results"]:
                        sv = result.get("SEARCHVAL", "")
                        if "MRT" in sv or "LRT" in sv:
                            result["STATION_CODE"] = code
                            mrt_lrt_data.append(result)
                            found = True
                            break

                if found:
                    break

                time.sleep(0.03)
            except Exception as e:
                logger.warning("Failed to fetch %s (search=%s): %s", code, search_val, e)

        if not found:
            logger.warning("Failed to find station for %s (%s)", code, station_name)
            failed_codes.append(code)

        time.sleep(0.05)

    state["raw_data"] = mrt_lrt_data
    state["quality_report"] = {
        "total_codes": len(state["station_codes"]),
        "extracted": len(mrt_lrt_data),
        "failed": len(failed_codes),
        "failed_codes": failed_codes[:20],
        "success_rate": round(len(mrt_lrt_data) / max(len(state["station_codes"]), 1) * 100, 2),
    }
    logger.info("Extracted %d / %d stations", len(mrt_lrt_data), len(state["station_codes"]))
    return state


def validate_extraction(state: PipelineState) -> PipelineState:
    """Agent reviews extraction quality and decides whether to proceed, retry, or abort."""
    qr = state["quality_report"]

    response = llm.invoke([
        SystemMessage(content=(
            "You are a data quality agent. Evaluate the extraction results and respond "
            "ONLY with valid JSON: "
            '{"decision": "proceed"|"retry"|"abort", "reasoning": "..."}'
            "\nRules: success_rate >= 80% → proceed; 50-80% → retry (max 2); <50% → abort."
        )),
        HumanMessage(content=(
            f"Extraction report: {json.dumps(qr)}. "
            f"Current retry count: {state['retry_count']}. "
            f"Max retries: 2. What should we do?"
        )),
    ])

    try:
        agent_output = json.loads(response.content)
    except json.JSONDecodeError:
        agent_output = {
            "decision": "proceed" if qr["success_rate"] >= 80 else "abort",
            "reasoning": "Fallback rule-based decision",
        }

    state["agent_decision"] = agent_output["decision"]
    state["agent_reasoning"] = agent_output["reasoning"]

    if agent_output["decision"] == "retry":
        state["retry_count"] += 1

    logger.info("Agent QA Decision: %s — %s", agent_output["decision"], agent_output["reasoning"])
    return state


def transform_data(state: PipelineState) -> PipelineState:
    schema = StructType([
        StructField("SEARCHVAL", StringType(), True),
        StructField("BLK_NO", StringType(), True),
        StructField("ROAD_NAME", StringType(), True),
        StructField("BUILDING", StringType(), True),
        StructField("ADDRESS", StringType(), True),
        StructField("POSTAL", StringType(), True),
        StructField("X", StringType(), True),
        StructField("Y", StringType(), True),
        StructField("LATITUDE", StringType(), True),
        StructField("LONGITUDE", StringType(), True),
        StructField("STATION_CODE", StringType(), True),
    ])

    df = spark.createDataFrame(state["raw_data"], schema=schema)
    df = df.dropDuplicates()
    df = df.drop("SEARCHVAL", "BLK_NO", "X", "Y")
    df = (
        df.withColumnRenamed("ROAD_NAME", "roadName")
          .withColumnRenamed("BUILDING", "building")
          .withColumnRenamed("ADDRESS", "address")
          .withColumnRenamed("POSTAL", "postalCode")
          .withColumnRenamed("LATITUDE", "latitude")
          .withColumnRenamed("LONGITUDE", "longitude")
          .withColumnRenamed("STATION_CODE", "stationCode")
    )
    df = df.withColumn("latitude", F.col("latitude").cast(DoubleType()))
    df = df.withColumn("longitude", F.col("longitude").cast(DoubleType()))

    # Derive MRT line from station code
    df = df.withColumn(
        "line",
        F.when(F.col("stationCode").rlike("^NS"), "North South Line")
         .when(F.col("stationCode").rlike("^NE"), "North East Line")
         .when(F.col("stationCode").rlike("^EW|^CG"), "East West Line")
         .when(F.col("stationCode").rlike("^CC|^CE"), "Circle Line")
         .when(F.col("stationCode").rlike("^TE"), "Thomson-East Coast Line")
         .when(F.col("stationCode").rlike("^DT"), "Downtown Line")
         .when(F.col("stationCode").rlike("^BP"), "Bukit Panjang LRT")
         .when(F.col("stationCode").rlike("^SE|^SW|^STC"), "Sengkang LRT")
         .when(F.col("stationCode").rlike("^PE|^PW|^PTC"), "Punggol LRT")
         .otherwise("Unknown")
    )

    state["transformed_data"] = [row.asDict() for row in df.collect()]
    logger.info("Transformed %d station records", df.count())
    return state


def decide_output(state: PipelineState) -> PipelineState:
    """Write cleaned MRT/LRT station data to same directory as this script."""
    output_dir = os.path.join(os.getcwd(), "dataset/transport")
    os.makedirs(output_dir, exist_ok=True)

    path = os.path.join(output_dir, "mrt_lrt_stations.csv")

    df = spark.createDataFrame(state["transformed_data"])
    df.toPandas().to_csv(path, index=False)

    state["output_path"] = path
    state["agent_reasoning"] = "CSV output to dataset/transport."
    logger.info("Output written to %s", path)
    return state


def route_after_plan(state: PipelineState) -> Literal["extract_data", "__end__"]:
    if state["agent_decision"] == "abort":
        logger.warning("Agent aborted pipeline: %s", state["agent_reasoning"])
        return END
    return "extract_data"


def route_after_validation(state: PipelineState) -> Literal["transform_data", "extract_data", "__end__"]:
    if state["agent_decision"] == "proceed":
        return "transform_data"
    elif state["agent_decision"] == "retry" and state["retry_count"] <= 2:
        return "extract_data"
    else:
        logger.warning("Agent aborted after validation: %s", state["agent_reasoning"])
        return END


workflow = StateGraph(PipelineState)

workflow.add_node("plan_extraction", plan_extraction)
workflow.add_node("extract_data", extract_data)
workflow.add_node("validate_extraction", validate_extraction)
workflow.add_node("transform_data", transform_data)
workflow.add_node("decide_output", decide_output)

workflow.set_entry_point("plan_extraction")
workflow.add_conditional_edges("plan_extraction", route_after_plan)
workflow.add_edge("extract_data", "validate_extraction")
workflow.add_conditional_edges("validate_extraction", route_after_validation)
workflow.add_edge("transform_data", "decide_output")
workflow.add_edge("decide_output", END)

app = workflow.compile()

# Section 2: HDB Data Loading & Exploration

# Load data set

Modeled from DataGovSG sample code

In [7]:
import os
import time
from pathlib import Path

import requests
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType
)


def load_secret(name: str, *alt_names: str) -> str:
    """
    Colab Secrets, or environment / `.env` when running locally.
    """
    try:
        from google.colab import userdata

        val = userdata.get(name)
        if val:
            return val
    except ImportError:
        pass
    try:
        from dotenv import load_dotenv

        load_dotenv(Path.cwd() / ".env")
    except ImportError:
        pass
    for key in (name, *alt_names):
        v = os.environ.get(key)
        if v:
            return v
    raise ValueError(
        f"Missing secret {name!r}. "
        "On Colab: add it under Secrets. Locally: set the env var or add it to `.env`."
    )


API_KEY = load_secret("GOV_DATA")

PRICE_DATASETS = [
    {"id": "d_8b84c4ee58e3cfc0ece0d773c8ca6abc", "label": "2017-present"},
    {"id": "d_ea9ed51da2787afaf8e51f827c304208", "label": "2015-2016"},
    {"id": "d_2d5ff9ea31397b66239f245f57751537", "label": "Mar2012-2014"},
    {"id": "d_43f493c6c50d54243cc1eab0df142d6a", "label": "2000-Feb2012"},
]

# fixed schema
PRICE_SCHEMA = StructType([
    StructField("month",               StringType(),  True),  # YYYY-MM
    StructField("town",                StringType(),  True),
    StructField("flat_type",           StringType(),  True),
    StructField("block",               StringType(),  True),
    StructField("street_name",         StringType(),  True),
    StructField("storey_range",        StringType(),  True),  # e.g. "07 TO 09"
    StructField("floor_area_sqm",      DoubleType(),  True),
    StructField("flat_model",          StringType(),  True),
    StructField("lease_commence_date", IntegerType(), True),  # YYYY
    StructField("remaining_lease",     StringType(),  True),  # e.g. "61 years 04 months"
    StructField("resale_price",        DoubleType(),  True),
    StructField("_source",             StringType(),  True),
])

def align_to_schema(df, label):
    """
    Align any dataset to the canonical schema regardless of which
    columns are present. Missing columns are added as null.
    Casts are applied after alignment so types are always consistent.
    """

    # Normalise column names to lowercase + underscores to match schema
    for col in df.columns:
        clean = col.strip().lower().replace(" ", "_")
        if clean != col:
            df = df.withColumnRenamed(col, clean)

    # Add missing columns as null
    canonical_cols = [
        "month", "town", "flat_type", "block", "street_name",
        "storey_range", "floor_area_sqm", "flat_model",
        "lease_commence_date", "remaining_lease", "resale_price"
    ]
    for col in canonical_cols:
        if col not in df.columns:
            print(f"    [{label}] '{col}' not found — adding as null")
            df = df.withColumn(col, F.lit(None).cast(StringType()))

    # Cast numeric columns safely after all columns are present
    df = (
        df
        .withColumn("floor_area_sqm",
                    F.when(F.col("floor_area_sqm").rlike(r"^-?\d+(\.\d+)?$"),
                           F.col("floor_area_sqm").cast(DoubleType()))
                    .otherwise(F.lit(None).cast(DoubleType())))
        .withColumn("lease_commence_date",
                    F.when(F.col("lease_commence_date").rlike(r"^\d+$"),
                           F.col("lease_commence_date").cast(IntegerType()))
                    .otherwise(F.lit(None).cast(IntegerType())))
        .withColumn("resale_price",
                    F.when(F.col("resale_price").rlike(r"^-?\d+(\.\d+)?$"),
                           F.col("resale_price").cast(DoubleType()))
                    .otherwise(F.lit(None).cast(DoubleType())))
    )

    # Keep only canonical columns + source tag in consistent order
    df = df.select(canonical_cols).withColumn("_source", F.lit(label))
    return df


s = requests.Session()
s.headers.update({
    "referer":   "https://colab.research.google.com",
    "x-api-key": API_KEY,
})

def download_dataset(dataset_id, label, schema, max_polls=10):
    print(f"  [{label}]  initiating download...")

    s.get(
        f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/initiate-download",
        json={}
    )

    for i in range(max_polls):
        poll = s.get(
            f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/poll-download",
            json={}
        ).json()

        if "url" in poll["data"]:
            url = poll["data"]["url"]
            print(f"    ready, downloading...")

            lines = requests.get(url).text.splitlines()
            rdd   = spark.sparkContext.parallelize(lines)
            df    = (spark.read
                         .option("header", "true")
                         .csv(rdd))
            df = df.withColumn("_source", F.lit(label))

            # Align to canonical schema — adds missing columns as null,
            # casts existing columns to correct types
            df = align_to_schema(df, label)
            print(f"    done ✓  ({df.count():,} rows)")
            return df

        print(f"    poll {i+1}/{max_polls}: not ready yet...")
        time.sleep(3)

    raise TimeoutError(f"Dataset {label} never became ready after {max_polls} polls")

def load_from_api(dfs, schema):
    dfs = [download_dataset(ds["id"], ds["label"], schema) for ds in dfs]

    # Union all 4 DataFrames
    from functools import reduce
    combined = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)

    print(f"\n  Total rows loaded: {combined.count():,}")
    return combined

In [8]:
prices_df = load_from_api(PRICE_DATASETS,PRICE_SCHEMA)

  [2017-present]  initiating download...
    ready, downloading...
    done ✓  (227,932 rows)
  [2015-2016]  initiating download...
    ready, downloading...
    done ✓  (37,153 rows)
  [Mar2012-2014]  initiating download...
    ready, downloading...
    [Mar2012-2014] 'remaining_lease' not found — adding as null
    done ✓  (52,203 rows)
  [2000-Feb2012]  initiating download...
    ready, downloading...
    [2000-Feb2012] 'remaining_lease' not found — adding as null
    done ✓  (369,651 rows)

  Total rows loaded: 686,939


In [9]:
prices_df.printSchema()

root
 |-- month: string (nullable = true)
 |-- town: string (nullable = true)
 |-- flat_type: string (nullable = true)
 |-- block: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- storey_range: string (nullable = true)
 |-- floor_area_sqm: double (nullable = true)
 |-- flat_model: string (nullable = true)
 |-- lease_commence_date: integer (nullable = true)
 |-- remaining_lease: string (nullable = true)
 |-- resale_price: double (nullable = true)
 |-- _source: string (nullable = false)



In [10]:
prices_df.filter(F.col("_source") == "2000-Feb2012").show(10)

+-------+----------+---------+-----+----------------+------------+--------------+--------------+-------------------+---------------+------------+------------+
|  month|      town|flat_type|block|     street_name|storey_range|floor_area_sqm|    flat_model|lease_commence_date|remaining_lease|resale_price|     _source|
+-------+----------+---------+-----+----------------+------------+--------------+--------------+-------------------+---------------+------------+------------+
|2000-01|ANG MO KIO|   3 ROOM|  170|ANG MO KIO AVE 4|    07 TO 09|          69.0|      Improved|               1986|           NULL|    147000.0|2000-Feb2012|
|2000-01|ANG MO KIO|   3 ROOM|  174|ANG MO KIO AVE 4|    04 TO 06|          61.0|      Improved|               1986|           NULL|    144000.0|2000-Feb2012|
|2000-01|ANG MO KIO|   3 ROOM|  216|ANG MO KIO AVE 1|    07 TO 09|          73.0|New Generation|               1976|           NULL|    159000.0|2000-Feb2012|
|2000-01|ANG MO KIO|   3 ROOM|  215|ANG MO KIO

# Geocode

Geocode HDB flat addresses using OneMap API to obtain lat/long coordinates

In [11]:
import requests
from google.colab import userdata

url = "https://www.onemap.gov.sg/api/auth/post/getToken"

email = userdata.get("ONEMAP_EMAIL")
password = userdata.get("ONEMAP_PASSWORD")

url = "https://www.onemap.gov.sg/api/auth/post/getToken"
payload = {
    "email": email,
    "password": password
}

response = requests.post(url, json=payload)
data = response.json()

access_token = data['access_token']
access_token

def geocode_onemap(address):
    url = "https://www.onemap.gov.sg/api/common/elastic/search"

    params = {
        "searchVal": address,
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }

    try:
        response = requests.get(url, params=params, timeout=5)
        data = response.json()

        if data["found"] > 0:
            result = data["results"][0]
            return float(result["LATITUDE"]), float(result["LONGITUDE"])
        else:
            return None, None
    except:
        return None, None

In [12]:
from pyspark.sql.functions import concat_ws
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType, StructType, StructField

# UDF that returns latitude and longitude as a struct
def geocode_udf(address):
    lat, lng = geocode_onemap(address)
    return (lat, lng)

# Define the schema of the UDF output
schema = StructType([
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True)
])

geocode_spark_udf = udf(geocode_udf, schema)
prices_df = prices_df.withColumn("full_address", concat_ws(" ", prices_df.block, prices_df.street_name))
prices_df = prices_df.withColumn("geo", geocode_spark_udf(prices_df.full_address))

# Split the struct into separate columns
prices_df = prices_df.withColumn("latitude", prices_df.geo.latitude).withColumn("longitude", prices_df.geo.longitude)
prices_df = prices_df.drop("geo")

prices_df.show(10)

+-------+----------+---------+-----+-----------------+------------+--------------+--------------+-------------------+------------------+------------+------------+--------------------+----------------+----------------+
|  month|      town|flat_type|block|      street_name|storey_range|floor_area_sqm|    flat_model|lease_commence_date|   remaining_lease|resale_price|     _source|        full_address|        latitude|       longitude|
+-------+----------+---------+-----+-----------------+------------+--------------+--------------+-------------------+------------------+------------+------------+--------------------+----------------+----------------+
|2017-01|ANG MO KIO|   2 ROOM|  406|ANG MO KIO AVE 10|    10 TO 12|          44.0|      Improved|               1979|61 years 04 months|    232000.0|2017-present|406 ANG MO KIO AV...|1.36200453938712|103.853879910407|
|2017-01|ANG MO KIO|   3 ROOM|  108| ANG MO KIO AVE 4|    01 TO 03|          67.0|New Generation|               1978|60 years 07

# Section 3: Data Cleaning Agent

# cleaning agent





In [13]:
!pip install -q langgraph langchain langchain-openai matplotlib seaborn

In [14]:
import json
import openai
from pyspark.sql.types import (
    StringType, LongType,
    FloatType, DateType, TimestampType
)
NUMERIC_TYPES = (DoubleType, FloatType, IntegerType, LongType)

In [15]:
client = openai.OpenAI(api_key=load_secret("BT4221_OPENAI_API_KEY", "OPENAI_API_KEY"))
MAX_RETRIES = 2

CLEANING TOOLS

In [16]:
def _transaction_month_to_date(column_name):
    """
    Parse a transaction-month column whether values are YYYY-MM, YYYY-MM-DD,
    or already Date/Timestamp. Avoids appending '-01' to a full date (which
    yields invalid strings like '2017-01-01-01').
    """
    s = F.trim(F.col(column_name).cast("string"))
    return (
        F.when(s.rlike(r"^\d{4}-\d{2}-\d{2}$"), F.to_date(s, "yyyy-MM-dd"))
        .when(s.rlike(r"^\d{4}-\d{2}$"), F.to_date(F.concat(s, F.lit("-01")), "yyyy-MM-dd"))
        .otherwise(F.try_to_date(s))
    )


def tool_trim_uppercase(sdf, column, **kwargs):
    """Trim whitespace and upper-case a string column."""
    return sdf.withColumn(column, F.upper(F.trim(F.col(column))))

def tool_parse_yyyymm_date(sdf, column, **kwargs):
    """
    Parse a YYYY-MM (or YYYY-MM-DD) column into DateType (first of month).
    Replaces the original column in-place.
    """
    return sdf.withColumn(column, _transaction_month_to_date(column))

def tool_parse_yyyymmdd_date(sdf, column, **kwargs):
    """Parse a YYYY-MM-DD string column into a DateType column."""
    return sdf.withColumn(column, F.to_date(F.col(column), "yyyy-MM-dd"))

def tool_parse_lease_string(sdf, column, **kwargs):
    """
    Parse remaining_lease string → numeric years, replacing the original column.
    Handles: '61 years 04 months', '62 years 01 month', '63 years'
    """
    years = F.nullif(
        F.regexp_extract(F.col(column), r"(\d+)\s*[Yy]ear", 1), F.lit("")
    ).cast(DoubleType())

    months = F.coalesce(
        F.nullif(F.regexp_extract(F.col(column), r"(\d+)\s*[Mm]onth", 1), F.lit(""))
         .cast(DoubleType()),
        F.lit(0.0)
    )
    # Replace original column
    return sdf.withColumn(column, F.coalesce(years, F.lit(None).cast(DoubleType())) + months / 12.0)

def tool_drop_nulls(sdf, column, **kwargs):
    """Drop rows where the specified column is null or empty string."""
    return sdf.filter(
        F.col(column).isNotNull() & (F.trim(F.col(column).cast("string")) != "")
    )

def tool_flag_outliers_zscore(sdf, column, group_by=None, threshold=4.0, **kwargs):
    """
    Add a boolean column {column}_outlier flagging rows where the z-score
    of `column` exceeds `threshold` (default 4.0).
    If group_by is provided, z-score is computed within each group.
    Rows are flagged, NOT removed.
    """
    out_col = column + "_outlier"
    w = Window.partitionBy(group_by) if group_by else Window.rowsBetween(
        Window.unboundedPreceding, Window.unboundedFollowing
    )
    return (
        sdf
        .withColumn("_mean", F.mean(column).over(w))
        .withColumn("_std",  F.stddev(column).over(w))
        .withColumn(
            out_col,
            F.when(
                F.col("_std") > 0,
                F.abs(F.col(column) - F.col("_mean")) / F.col("_std") > threshold
            ).otherwise(F.lit(False))
        )
        .drop("_mean", "_std")
    )

def tool_standardise_values(sdf, column, replacements, **kwargs):
    """
    Replace inconsistent string values in a column.
    `replacements` is a dict: {"old_value": "new_value", ...}
    e.g. {"MULTI GENERATION": "MULTI-GENERATION", "MULTI-GEN": "MULTI-GENERATION"}
    """
    for old, new in replacements.items():
        sdf = sdf.withColumn(
            column,
            F.when(F.col(column) == old, F.lit(new)).otherwise(F.col(column))
        )
    return sdf

def tool_impute_remaining_lease(sdf, column, **kwargs):
    """
    Impute missing remaining_lease values using:
        99 - (transaction_year - lease_commence_date)
    Only fills nulls — existing values are preserved.
    Requires: lease_commence_date (int) and month (YYYY-MM string) to be present.
    """
    sale_year = F.year(_transaction_month_to_date("month"))
    approximated = (99 - (sale_year - F.col("lease_commence_date"))).cast(DoubleType())

    return sdf.withColumn(
        column,
        F.coalesce(F.col(column), approximated)   # keep existing, fill nulls
    )
def tool_parse_storey_range(sdf, column, **kwargs):
    """
    Parse 'XX TO YY' storey_range string into a numeric storey_midpoint column.
    e.g. "04 TO 06" → storey_midpoint = 5.0
    """
    lo = F.regexp_extract(F.col(column), r"^(\d+)", 1).cast("double")
    hi = F.regexp_extract(F.col(column), r"(\d+)$", 1).cast("double")
    return sdf.withColumn("storey_midpoint", (lo + hi) / 2)


def tool_derive_year_month(sdf, column, **kwargs):
    """
    Extract transaction_year (int) and transaction_month (int) from a YYYY-MM column.
    Requires the column to be either a string 'YYYY-MM' or already a DateType.
    """
    date_col = _transaction_month_to_date(column)
    return (sdf
        .withColumn("transaction_year",  F.year(date_col).cast("int"))
        .withColumn("transaction_month", F.month(date_col).cast("int"))
    )


def tool_flag_outliers_iqr(sdf, column, multiplier=1.5, **kwargs):
    """
    Flag outliers using the IQR method. Adds a boolean column {column}_outlier.
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    Rows are flagged, NOT removed here (removal happens in post-processing).
    """
    quantiles = sdf.approxQuantile(column, [0.25, 0.75], 0.01)
    q1, q3 = quantiles[0], quantiles[1]
    iqr = q3 - q1
    lower = q1 - multiplier * iqr
    upper = q3 + multiplier * iqr
    out_col = f"{column}_outlier"
    return sdf.withColumn(
        out_col,
        (F.col(column) < lower) | (F.col(column) > upper)
    )


def tool_compute_remaining_lease(sdf, column, **kwargs):
    """
    Always compute remaining_lease_years from lease_commence_date.
    Formula: 99 - (transaction_year - lease_commence_date)
    Overwrites the column for ALL rows — no string parsing needed.
    Reliable for all 4 datasets including pre-2015 data where remaining_lease is null.
    Requires: transaction_year (int) and lease_commence_date (int) columns.
    Run AFTER derive_year_month.
    """
    computed = (99 - (F.col("transaction_year") - F.col("lease_commence_date"))).cast(DoubleType())
    return sdf.withColumn("remaining_lease_years", computed)


def tool_drop_column(sdf, column, **kwargs):
    """Drop a column nominated by the agent in columns_to_drop."""
    return sdf.drop(column)


In [17]:
# Registry — maps tool name (string) → function
TOOL_REGISTRY = {
    "trim_uppercase":         tool_trim_uppercase,
    "parse_yyyymm_date":      tool_parse_yyyymm_date,
    "parse_yyyymmdd_date":    tool_parse_yyyymmdd_date,
    "parse_lease_string":     tool_parse_lease_string,
    "drop_nulls":             tool_drop_nulls,
    "flag_outliers_zscore":   tool_flag_outliers_zscore,
    "flag_outliers_iqr":      tool_flag_outliers_iqr,
    "standardise_values":     tool_standardise_values,
    "impute_remaining_lease": tool_impute_remaining_lease,
    "parse_storey_range":     tool_parse_storey_range,
    "derive_year_month":      tool_derive_year_month,
    "compute_remaining_lease": tool_compute_remaining_lease,
    "drop_column":             tool_drop_column,
}

TOOL_CATALOGUE = """
Available cleaning tools (name → what it does → required args → optional args):
  trim_uppercase          → trim whitespace and upper-case a string column
                            args: column
  parse_yyyymm_date       → parse YYYY-MM string → DateType (first of month)
                            args: column
  parse_yyyymmdd_date     → parse YYYY-MM-DD string → DateType
                            args: column
  parse_lease_string      → extract years from '61 years 04 months' or plain '61'
                            → replaces column with double (years)
                            args: column
  drop_nulls              → drop rows where column is null or empty
                            args: column
  flag_outliers_iqr       → PREFERRED outlier method: add {col}_outlier boolean using IQR
                            lower = Q1 - multiplier*IQR, upper = Q3 + multiplier*IQR
                            args: column   optional: multiplier (float, default 1.5)
                            USE THIS for resale_price (required by project spec)
  flag_outliers_zscore    → add {col}_outlier boolean (z-score > threshold, default 4.0)
                            args: column   optional: group_by (column name), threshold (float)
  standardise_values      → replace inconsistent string values
                            args: column, replacements (dict of old→new)
  impute_remaining_lease  → fill null remaining_lease values using
                            99 - (sale_year - lease_commence_date)
                            args: column
                            requires: month and lease_commence_date columns
  parse_storey_range      → parse 'XX TO YY' string → numeric storey_midpoint column
                            args: column (must be storey_range)
  derive_year_month       → extract transaction_year (int) and transaction_month (int)
                            from a YYYY-MM column
                            args: column (must be month)
  compute_remaining_lease → always compute remaining_lease_years = 99 - (transaction_year - lease_commence_date)
                            args: column (ignored — always writes remaining_lease_years)
                            requires: transaction_year and lease_commence_date columns
                            run AFTER derive_year_month
  drop_column            → drop a column nominated by the agent (columns_to_drop list)
                            args: column
"""

PROFILER

dataset info for llm to plan

In [21]:
def profile_dataframe(sdf):
    total       = sdf.count()
    sample_rows = sdf.limit(3).collect()

    # Header
    lines = [
        f"Total rows: {total:,}",
        "",
        f"{'Column':<25} {'Type':<15} {'Nulls':>8} {'Distinct':>10}  {'Patterns':<30}  {'Samples'}",
        "-" * 110
    ]

    for field in sdf.schema.fields:
        col   = field.name
        dtype = str(field.dataType)

        # Nulls
        if isinstance(field.dataType, StringType):
            null_n = sdf.filter(F.col(col).isNull() | (F.trim(F.col(col)) == "")).count()
        else:
            null_n = sdf.filter(F.col(col).isNull()).count()
        null_str = f"{null_n/total*100:.0f}%" if null_n > 0 else "0%"

        # Distinct
        distinct_n = sdf.select(col).distinct().count()

        # Patterns (string only)
        patterns = ""
        if isinstance(field.dataType, StringType):
            non_null   = sdf.filter(F.col(col).isNotNull() & (F.trim(F.col(col)) != ""))
            non_null_n = non_null.count()
            if non_null_n > 0:
                def pct(pattern):
                    return non_null.filter(F.col(col).rlike(pattern)).count() / non_null_n * 100
                checks = [
                    ("YYYY-MM",   pct(r"^\d{4}-\d{2}$")),
                    ("X TO Y",    pct(r"^\d+ TO \d+$")),
                    ("unit str",  pct(r"\d+\s*(year|month)")),
                ]
                patterns = "  ".join(f"{label}:{v:.0f}%" for label, v in checks if v > 10)

        # Numeric stats
        if isinstance(field.dataType, NUMERIC_TYPES):
            s = sdf.select(F.min(col), F.max(col)).collect()[0]
            patterns = f"min={s[0]}  max={s[1]}"

        # Samples
        samples = ", ".join(str(r[col]) for r in sample_rows if r[col] is not None)[:50]

        lines.append(
            f"{col:<25} {dtype:<15} {null_str:>8} {distinct_n:>10}  {patterns:<30}  {samples}"
        )

    return "\n".join(lines)

LLM helper

In [19]:
def ask_llm_for_config(profile_str):
    """
    Ask the LLM to return a structured cleaning_config.
    The LLM decides the strategy; config_to_plan() translates it to tool calls,
    ensuring PySpark cleaning is driven by the agent's config — not hardcoded.
    """
    prompt = f"""You are a data cleaning expert for an HDB resale price ML pipeline.
Analyse the DataFrame profile and return a cleaning configuration JSON.

{TOOL_CATALOGUE}

DATAFRAME PROFILE:
{profile_str}

Return ONLY valid JSON with exactly these six keys:

{{
  "storey_range_strategy": "parse_storey_range",
  "outlier_method": "IQR",
  "outlier_threshold": <float, choose 1.5 to 3.0 based on price spread>,
  "null_handling": {{
    "<column_name>": "drop_nulls" | "impute_remaining_lease"
  }},
  "columns_to_drop": ["<col>"],
  "duplicate_strategy": "dropDuplicates"
}}

Rules:
  - storey_range_strategy: always "parse_storey_range"
  - outlier_method: always "IQR" (required by project spec)
  - outlier_threshold: between 1.5 and 3.0 — choose based on the resale_price spread
  - null_handling: only include columns that have >0 nulls in the profile;
    use "impute_remaining_lease" for remaining_lease, "drop_nulls" for others
  - columns_to_drop: list columns that add no predictive value (e.g. _source)
  - duplicate_strategy: always "dropDuplicates"
"""
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
    )
    raw = resp.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw)


def config_to_plan(cleaning_config):
    """
    Convert a cleaning_config dict into an ordered list of
    tool calls compatible with TOOL_REGISTRY.
    This is how PySpark cleaning is driven by the agent's config.
    """
    plan = []

    # 1. Trim + uppercase all string categoricals
    for col in ["town", "flat_type", "flat_model", "street_name", "block"]:
        plan.append({"tool": "trim_uppercase", "column": col, "kwargs": {},
                     "reason": "Normalise categorical text"})

    # 2. Parse month -> DateType, then extract integer year/month
    plan.append({"tool": "parse_yyyymm_date", "column": "month", "kwargs": {},
                 "reason": "Convert YYYY-MM string to DateType"})
    plan.append({"tool": "derive_year_month", "column": "month", "kwargs": {},
                 "reason": "Extract transaction_year and transaction_month"})

    # 3. Storey range -> midpoint (driven by config)
    if cleaning_config.get("storey_range_strategy") == "parse_storey_range":
        plan.append({"tool": "parse_storey_range", "column": "storey_range", "kwargs": {},
                     "reason": "Parse storey_range string to numeric storey_midpoint"})

    # 4. Flat type standardisation (canonical values)
    plan.append({
        "tool": "standardise_values", "column": "flat_type",
        "kwargs": {"replacements": {
            "MULTI GENERATION": "MULTI-GENERATION",
            "MULTI-GEN": "MULTI-GENERATION",
            "multi generation": "MULTI-GENERATION",
        }},
        "reason": "Standardise flat_type to canonical values"
    })

    # 5. Compute remaining_lease_years from lease_commence_date (reliable for all 4 datasets)
    # Replaces parse_lease_string + impute_remaining_lease — handles null columns for pre-2015 data
    plan.append({"tool": "compute_remaining_lease", "column": "remaining_lease", "kwargs": {},
                 "reason": "Compute remaining_lease_years = 99 - (transaction_year - lease_commence_date) for all rows"})
    # Drop original remaining_lease string column — always superseded by remaining_lease_years
    plan.append({"tool": "drop_column", "column": "remaining_lease", "kwargs": {},
                 "reason": "Drop original remaining_lease string column (replaced by remaining_lease_years)"})

    # 6. Null handling per config (agent decides which columns need drop_nulls)
    for col, strategy in cleaning_config.get("null_handling", {}).items():
        if strategy == "drop_nulls" and col != "remaining_lease":
            plan.append({"tool": "drop_nulls", "column": col, "kwargs": {},
                         "reason": f"Remove null rows in {col} (agent decision from cleaning_config)"})

    # 6b. Drop columns nominated by agent (columns_to_drop)
    for col in cleaning_config.get("columns_to_drop", []):
        plan.append({"tool": "drop_column", "column": col, "kwargs": {},
                     "reason": f"Drop {col} per agent config — no predictive value"})

    # 7. Outlier flagging driven by config method + threshold
    method    = cleaning_config.get("outlier_method", "IQR")
    threshold = float(cleaning_config.get("outlier_threshold", 1.5))
    if method == "IQR":
        plan.append({"tool": "flag_outliers_iqr", "column": "resale_price",
                     "kwargs": {"multiplier": threshold},
                     "reason": f"Flag resale_price outliers using IQR (multiplier={threshold}) per agent config"})
    else:
        plan.append({"tool": "flag_outliers_zscore", "column": "resale_price",
                     "kwargs": {"threshold": threshold},
                     "reason": f"Flag resale_price outliers using z-score (threshold={threshold}) per agent config"})

    return plan


def ask_llm_to_fix(error_type, error_msg, failed_call, profile_str):
    prompt = f"""A cleaning tool call has failed. Revise the arguments to fix it.

FAILED CALL:
{json.dumps(failed_call, indent=2)}

ERROR TYPE   : {error_type}
ERROR MESSAGE: {error_msg}

{TOOL_CATALOGUE}

DATAFRAME PROFILE:
{profile_str}

Rules:
  - Revise only the arguments of the failed call — do not change the intent
  - If the column type is wrong for the tool, suggest the correct tool instead
  - Do not suggest null handling tools — those are handled separately by the human
  - Do not suggest impute_remaining_lease unless the original call was already
    related to remaining_lease

Return ONLY valid JSON, no markdown:
{{
  "diagnosis": "one sentence root cause",
  "revised_call": {{
    "tool": "tool_name",
    "column": "column_name",
    "kwargs": {{}}
  }}
}}"""

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    raw = resp.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw)


PIPELINE

In [ ]:
print("\nStep 1: Profiling DataFrame...")
profile_str = profile_dataframe(prices_df)
print(profile_str)

# Null counts BEFORE cleaning
print("\n=== Null counts BEFORE cleaning ===")
prices_df.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in prices_df.columns]
).show(vertical=True)

print("\nStep 2: Asking LLM for cleaning configuration...")
cleaning_config = ask_llm_for_config(profile_str)
print("\nCleaning config (agent decision):")
print(json.dumps(cleaning_config, indent=2))

# Save cleaning_decisions.md immediately after LLM returns config
import datetime
with open("cleaning_decisions.md", "w") as f:
    f.write("# Cleaning Decisions\n\n")
    f.write(f"**Agent:** Data Cleaning Agent (gpt-4o-mini)\n")
    f.write(f"**Timestamp:** {datetime.datetime.now().isoformat()}\n\n")
    f.write("## Cleaning Config\n\n")
    f.write(f"```json\n{json.dumps(cleaning_config, indent=2)}\n```\n")
print("\ncleaning_decisions.md written ✓")

# Convert config to ordered tool-call plan (PySpark reads config via this function)
plan = config_to_plan(cleaning_config)
print(f"\nDerived plan — {len(plan)} tool calls:")
for i, call in enumerate(plan, 1):
    kwargs_str = f"  kwargs={call['kwargs']}" if call.get("kwargs") else ""
    print(f"  {i}. {call['tool']}({call['column']}){kwargs_str}")
    print(f"     → {call['reason']}")

fix_log     = []
current_sdf = prices_df

print("\nStep 3: Executing plan...")
print("=" * 52)

for call in plan:
    tool_name    = call["tool"]
    column       = call["column"]
    kwargs       = call.get("kwargs", {})
    current_call = call.copy()

    print(f"\n  {tool_name}({column})")

    if tool_name not in TOOL_REGISTRY:
        print(f"  SKIPPED - unknown tool '{tool_name}'")
        continue

    success = False

    for attempt in range(1, MAX_RETRIES + 2):
        try:
            result_sdf = TOOL_REGISTRY[tool_name](current_sdf, column, **kwargs)
            result_sdf.count()   # force Spark evaluation

            print(f"  Attempt {attempt}: PASSED")
            current_sdf = result_sdf
            success = True
            break

        except Exception as e:
            err_type = type(e).__name__
            err_msg  = str(e)[:300]
            print(f"  Attempt {attempt}: FAILED  ({err_type}: {err_msg[:80]})")

            if attempt > MAX_RETRIES:
                print("  Max retries reached. Skipping.")
                break

            print("  Asking LLM to revise call...")
            llm_result   = ask_llm_to_fix(err_type, err_msg, current_call, profile_str)
            diagnosis    = llm_result.get("diagnosis", "Unknown")
            revised      = llm_result.get("revised_call", {})
            tool_name    = revised.get("tool", tool_name)
            column       = revised.get("column", column)
            kwargs       = revised.get("kwargs", kwargs)
            current_call = revised

            print(f"  Diagnosis    : {diagnosis}")
            print(f"  Revised call : {tool_name}({column}) kwargs={kwargs}")

            fix_log.append({
                "original": call,
                "attempt":  attempt,
                "error":    err_msg[:150],
                "diagnosis": diagnosis,
                "revised":  revised,
            })

    if not success:
        print(f"  Could not complete after {MAX_RETRIES} retries.")



Step 1: Profiling DataFrame...


In [ ]:
print("PIPELINE FINISHED")
print(f"Total automatic revisions: {len(fix_log)}")

print("\nFINAL SCHEMA")
current_sdf.printSchema()

print("\nFINAL SAMPLE")
current_sdf.show(5, truncate=False)

if fix_log:
    print("\nFIX LOG")
    print("-" * 52)
    for i, fix in enumerate(fix_log, 1):
        orig = fix["original"]
        print(f"\nRevision {i} — {orig['tool']}({orig['column']})  attempt {fix['attempt']}")
        print(f"  Error    : {fix['error']}")
        print(f"  Diagnosis: {fix['diagnosis']}")
        print(f"  Revised  : {fix['revised']}")

clean_df = current_sdf
print(f"\nclean_df ready: {clean_df.count():,} rows, {len(clean_df.columns)} columns")

# ── POST-PROCESSING (deterministic) ─────────────────────────────────────────

# 0. Deterministic flat_type whitelist validation
CANONICAL_FLAT_TYPES = {"1 ROOM", "2 ROOM", "3 ROOM", "4 ROOM", "5 ROOM", "EXECUTIVE", "MULTI-GENERATION"}
n_invalid = clean_df.filter(~F.col("flat_type").isin(list(CANONICAL_FLAT_TYPES))).count()
if n_invalid > 0:
    print(f"\nWarning: {n_invalid:,} rows with non-canonical flat_type — filtering out")
    clean_df = clean_df.filter(F.col("flat_type").isin(list(CANONICAL_FLAT_TYPES)))
else:
    print("flat_type: all values are canonical ✓")

# 1. Validate floor_area_sqm — no negative values
n_neg = clean_df.filter(F.col("floor_area_sqm") <= 0).count()
if n_neg > 0:
    print(f"Removing {n_neg:,} rows with non-positive floor_area_sqm")
    clean_df = clean_df.filter(F.col("floor_area_sqm") > 0)

# 2. Remove exact duplicates
n_before_dedup = clean_df.count()
clean_df = clean_df.dropDuplicates()
n_after_dedup = clean_df.count()
print(f"\nDuplicates removed: {n_before_dedup - n_after_dedup:,} rows ({n_before_dedup:,} → {n_after_dedup:,})")

# 3. Remove outliers flagged by IQR
outlier_col = "resale_price_outlier"
n_removed = 0
if outlier_col in clean_df.columns:
    n_before_out = clean_df.count()
    clean_df = clean_df.filter(~F.col(outlier_col)).drop(outlier_col)
    n_after_out = clean_df.count()
    n_removed = n_before_out - n_after_out
    print(f"Outliers removed  : {n_removed:,} rows ({n_removed/n_before_out*100:.1f}%) — IQR method on resale_price")

# 4. Null counts AFTER cleaning
print("\n=== Null counts AFTER cleaning ===")
clean_df.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in clean_df.columns]
).show(vertical=True)

# 5. Consolidated cleaning summary
n_raw = prices_df.count()  # actual raw row count
n_final = clean_df.count()
print("\n=== Cleaning Summary (Before vs After) ===")
print(f"{'Stage':<35} {'Rows':>10}  {'Change':>20}")
print("-" * 68)
print(f"{'Raw rows loaded':<35} {n_raw:>10,}  {'—':>20}")
print(f"{'After deduplication':<35} {n_after_dedup:>10,}  {f'-{n_before_dedup - n_after_dedup:,} dupes':>20}")
print(f"{'After outlier removal':<35} {n_final:>10,}  {f'-{n_removed:,} outliers':>20}")
print(f"{'Total rows removed':<35} {n_raw - n_final:>10,}  {f'{(n_raw - n_final)/n_raw*100:.1f}% of raw':>20}")
print(f"\nclean_df final: {n_final:,} rows, {len(clean_df.columns)} columns")


# EDA

In [ ]:
# ── EDA 1: Price trends over time ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

df = clean_df

trend_pdf = (
    df
      .groupBy("month", "flat_type")
      .agg(F.percentile_approx("resale_price", 0.5).alias("median_price"))
      .toPandas()
)

trend_pdf["month"] = pd.to_datetime(trend_pdf["month"])
trend_pdf = trend_pdf.sort_values("month")

trend_pdf["median_price"] = (
    trend_pdf.groupby("flat_type")["median_price"]
             .transform(lambda x: x.rolling(6, min_periods=1).mean())
)

fig, ax = plt.subplots(figsize=(13,5))

for ft, grp in trend_pdf.groupby("flat_type"):
    ax.plot(grp["month"], grp["median_price"]/1e3, linewidth=2, label=ft)

import matplotlib.dates as mdates
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax.set_title("Median HDB Resale Price Trend by Flat Type", fontsize=14, fontweight="bold")
ax.set_ylabel("Median Price (SGD '000)")
ax.set_xlabel("Year")

ax.legend(title="Flat Type", bbox_to_anchor=(1.02,1))
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── EDA 2: Price distribution by flat type & town ──
dist_pdf = (
    df
      .select("resale_price", "flat_type", "town")
      .toPandas()
)

# Violin by flat_type
fig, ax = plt.subplots(figsize=(12, 5))
order = dist_pdf.groupby("flat_type")["resale_price"].median().sort_values().index
sns.violinplot(data=dist_pdf, x="flat_type", y="resale_price",
               order=order, palette="muted", ax=ax, inner="quartile")
ax.set_title("Resale Price Distribution by Flat Type", fontsize=13, fontweight="bold")
ax.set_xlabel("Flat Type")
ax.set_ylabel("Resale Price (SGD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}k"))
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

# Box by top-10 towns
top10 = dist_pdf.groupby("town")["resale_price"].median().nlargest(10).index
fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(data=dist_pdf[dist_pdf["town"].isin(top10)], x="town", y="resale_price",
            order=top10, palette="Set2", ax=ax, flierprops={"markersize": 2})
ax.set_title("Resale Price Distribution — Top 10 Towns by Median Price",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Town")
ax.set_ylabel("Resale Price (SGD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}k"))
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ── EDA 3: Correlation heatmap ──
corr_cols = [c for c in [
    "resale_price", "floor_area_sqm", "lease_commence_date",
    "storey_midpoint", "transaction_year", "transaction_month", "remaining_lease_years"
] if c in df.columns]

corr_pdf = df.select(corr_cols).dropna().limit(50_000).toPandas()
corr_pdf = corr_pdf.select_dtypes(include="number")
corr_mat = corr_pdf.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, mask=mask, annot=True, fmt=".2f",
            cmap="RdYlGn", center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, annot_kws={"size": 9})
ax.set_title("Feature Correlation Heatmap", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n  Correlations with resale_price (absolute, descending):")
print(
    corr_mat["resale_price"]
    .drop("resale_price")
    .abs()
    .sort_values(ascending=False)
    .to_string()
)

End

# Section 4: ML Pipeline (LangGraph)

In [ ]:
import os
import json
import logging
from datetime import datetime
from typing import TypedDict

from dotenv import load_dotenv
from langgraph.graph import StateGraph, END, START
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv()

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
)


# ---------------------------------------------------------------------------
# AgentState
# ---------------------------------------------------------------------------

class AgentState(TypedDict):
    """Shared state flowing through the LangGraph pipeline."""

    # Input: dataset metadata populated before pipeline starts
    dataset_info: dict          # schema, row count, column types, null counts, sample values

    # Cleaning stage
    cleaning_config: dict       # JSON from cleaning agent (outlier_method, null_handling, etc.)
    cleaned_data_summary: dict  # column stats, correlations, sample after cleaning

    # Feature engineering stage
    feature_config: dict        # JSON from FE agent (distance_features, time_features, etc.)
    feature_summary: dict       # summary of features created

    # Model training stage
    model_results: dict         # RMSE, MAE, R², MAPE per model

    # Evaluation stage
    evaluation_report: dict     # best_model, reasoning, key_findings, recommendations

    # Logging
    agent_logs: list            # append-only log of agent decisions and timestamps


# ---------------------------------------------------------------------------
# Helper
# ---------------------------------------------------------------------------

def _log_entry(agent_name: str, detail: str) -> dict:
    return {
        "agent": agent_name,
        "timestamp": datetime.now().isoformat(),
        "detail": detail,
    }


# ---------------------------------------------------------------------------
# Node 1: cleaning_agent
# ---------------------------------------------------------------------------

def cleaning_agent(state: AgentState) -> dict:
    """LLM agent that produces structured cleaning decisions from dataset_info."""
    logger.info("Node: cleaning_agent")

    # TODO: implement full agent skill in US-5.3
    # Expected: call OpenAI with dataset_info, return JSON with keys:
    #   storey_range_strategy, outlier_method, outlier_threshold,
    #   null_handling, columns_to_drop, duplicate_strategy

    cleaning_config = {
        "storey_range_strategy": "midpoint",
        "outlier_method": "IQR",
        "outlier_threshold": 1.5,
        "null_handling": {"default": "drop"},
        "columns_to_drop": [],
        "duplicate_strategy": "drop_exact",
    }

    logs = list(state.get("agent_logs", []))
    logs.append(_log_entry("cleaning_agent", "Produced cleaning config (stub)"))

    return {"cleaning_config": cleaning_config, "agent_logs": logs}


# ---------------------------------------------------------------------------
# Node 2: apply_cleaning
# ---------------------------------------------------------------------------

def apply_cleaning(state: AgentState) -> dict:
    """PySpark node that executes cleaning based on cleaning_config."""
    logger.info("Node: apply_cleaning")

    config = state["cleaning_config"]

    # TODO: implement PySpark cleaning logic
    # Read config["outlier_method"], config["null_handling"], etc.
    # Apply to Spark DataFrame (accessed via module-level reference)

    cleaned_data_summary = {
        "rows_before": 0,
        "rows_after": 0,
        "columns": [],
        "null_counts": {},
        "outliers_removed": 0,
        "duplicates_removed": 0,
    }

    return {"cleaned_data_summary": cleaned_data_summary}


# ---------------------------------------------------------------------------
# Node 3: feature_engineering_agent
# ---------------------------------------------------------------------------

def feature_engineering_agent(state: AgentState) -> dict:
    """LLM agent that recommends feature engineering from cleaned data summary."""
    logger.info("Node: feature_engineering_agent")

    # TODO: implement full agent skill in US-5.4
    # Expected: call OpenAI with cleaned_data_summary, return JSON with keys:
    #   distance_features, time_features, categorical_features,
    #   binning, interaction_features, features_to_drop

    feature_config = {
        "distance_features": [],
        "time_features": [],
        "categorical_features": [],
        "binning": {},
        "interaction_features": [],
        "features_to_drop": [],
    }

    logs = list(state.get("agent_logs", []))
    logs.append(_log_entry("feature_engineering_agent", "Produced feature config (stub)"))

    return {"feature_config": feature_config, "agent_logs": logs}


# ---------------------------------------------------------------------------
# Node 4: apply_feature_engineering
# ---------------------------------------------------------------------------

def apply_feature_engineering(state: AgentState) -> dict:
    """PySpark node that creates features based on feature_config."""
    logger.info("Node: apply_feature_engineering")

    config = state["feature_config"]

    # TODO: implement PySpark feature engineering logic
    # Read config and create distance, time, categorical, binning,
    # interaction features on the Spark DataFrame

    feature_summary = {
        "features_created": [],
        "features_dropped": [],
        "final_feature_count": 0,
    }

    return {"feature_summary": feature_summary}


# ---------------------------------------------------------------------------
# Node 5: train_models
# ---------------------------------------------------------------------------

def train_models(state: AgentState) -> dict:
    """Train ML models and compute evaluation metrics."""
    logger.info("Node: train_models")

    # TODO: implement PySpark MLlib training
    # Train 3 models (e.g. LinearRegression, RandomForestRegressor, GBTRegressor)
    # Compute RMSE, MAE, R², MAPE for each

    model_results = {
        "LinearRegression": {"RMSE": 0.0, "MAE": 0.0, "R2": 0.0, "MAPE": 0.0},
        "RandomForest": {"RMSE": 0.0, "MAE": 0.0, "R2": 0.0, "MAPE": 0.0},
        "GBT": {"RMSE": 0.0, "MAE": 0.0, "R2": 0.0, "MAPE": 0.0},
    }

    return {"model_results": model_results}


# ---------------------------------------------------------------------------
# Node 6: evaluation_agent
# ---------------------------------------------------------------------------

def evaluation_agent(state: AgentState) -> dict:
    """LLM agent that produces a comparative evaluation of model results."""
    logger.info("Node: evaluation_agent")

    # TODO: implement full agent skill in US-5.5
    # Expected: call OpenAI with model_results, return JSON with keys:
    #   best_model, reasoning, key_findings, recommendations, model_ranking

    evaluation_report = {
        "best_model": "",
        "reasoning": "",
        "key_findings": [],
        "recommendations": [],
        "model_ranking": [],
    }

    logs = list(state.get("agent_logs", []))
    logs.append(_log_entry("evaluation_agent", "Produced evaluation report (stub)"))

    return {"evaluation_report": evaluation_report, "agent_logs": logs}


# ---------------------------------------------------------------------------
# Pipeline builder
# ---------------------------------------------------------------------------

def build_pipeline():
    """Build and compile the LangGraph pipeline with 6 nodes in linear sequence."""
    graph = StateGraph(AgentState)

    # Add nodes
    graph.add_node("cleaning_agent", cleaning_agent)
    graph.add_node("apply_cleaning", apply_cleaning)
    graph.add_node("feature_engineering_agent", feature_engineering_agent)
    graph.add_node("apply_feature_engineering", apply_feature_engineering)
    graph.add_node("train_models", train_models)
    graph.add_node("evaluation_agent", evaluation_agent)

    # Linear edges: START → cleaning_agent → ... → evaluation_agent → END
    graph.add_edge(START, "cleaning_agent")
    graph.add_edge("cleaning_agent", "apply_cleaning")
    graph.add_edge("apply_cleaning", "feature_engineering_agent")
    graph.add_edge("feature_engineering_agent", "apply_feature_engineering")
    graph.add_edge("apply_feature_engineering", "train_models")
    graph.add_edge("train_models", "evaluation_agent")
    graph.add_edge("evaluation_agent", END)

    return graph.compile()


# ---------------------------------------------------------------------------
# Main: build, visualize, and run
# ---------------------------------------------------------------------------

In [ ]:
spark.stop()
print('Spark session stopped.')